# 🤟 Gesture2Voice — Fingerspell Retrain (Custom Images)
Trains on YOUR personally collected images from webcam.

**Steps:** Install → Mount Drive → Unzip → Extract landmarks → Train → Download

In [ ]:
# Cell 1 — Install
!pip install mediapipe opencv-python-headless scikit-learn --quiet
!wget -q https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task -O hand_landmarker.task

import mediapipe as mp
print('MediaPipe version:', mp.__version__)
print('Model file ready:', __import__('os').path.exists('hand_landmarker.task'))
print('✅ Ready!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 5.8 MB/s eta 0:00:00
MediaPipe version: 0.10.35
Model file ready: True
✅ Ready!


In [ ]:
# Cell 2 — Mount Drive and unzip your dataset
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile

ZIP_PATH   = '/content/drive/MyDrive/Gesture2Voice/fingerspell_dataset.zip'
EXTRACT_TO = '/content/fingerspell_dataset'

if not os.path.exists(EXTRACT_TO):
    print('Unzipping...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall('/content/')
    print('✅ Unzipped')
else:
    print('✅ Already unzipped')

folders = sorted(os.listdir(EXTRACT_TO))
print(f'Found {len(folders)} classes: {folders}')

Mounted at /content/drive
Unzipping...
✅ Unzipped
Found 29 classes: ['A', 'B', 'C', 'D', 'DEL', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'NOTHING', 'O', 'P', 'Q', 'R', 'S', 'SPACE', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']


In [ ]:
# Cell 3 — Extract landmarks from your images
import cv2
import csv
import os
import mediapipe as mp

DATASET_DIR = '/content/fingerspell_dataset'
OUTPUT_CSV  = '/content/fingerspell_landmarks.csv'

BaseOptions           = mp.tasks.BaseOptions
HandLandmarker        = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
VisionRunningMode     = mp.tasks.vision.RunningMode

options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path='/content/hand_landmarker.task'),
    running_mode=VisionRunningMode.IMAGE,
    num_hands=1,
    min_hand_detection_confidence=0.2,
    min_hand_presence_confidence=0.2,
)

header = [f"{a}{i}" for i in range(21) for a in ['x','y','z']] + ['label']
total, skipped = 0, 0

with HandLandmarker.create_from_options(options) as landmarker:
    with open(OUTPUT_CSV, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(header)

        for label in sorted(os.listdir(DATASET_DIR)):
            label_path = os.path.join(DATASET_DIR, label)
            if not os.path.isdir(label_path):
                continue

            images = [f for f in os.listdir(label_path) if f.lower().endswith(('.jpg','.jpeg','.png'))]
            count  = 0

            for img_file in images:
                img_bgr = cv2.imread(os.path.join(label_path, img_file))
                if img_bgr is None:
                    skipped += 1
                    continue

                img_rgb  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
                mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
                result   = landmarker.detect(mp_image)

                if not result.hand_landmarks:
                    skipped += 1
                    continue

                lm = result.hand_landmarks[0]
                wx, wy, wz = lm[0].x, lm[0].y, lm[0].z
                row = []
                for pt in lm:
                    row.extend([pt.x - wx, pt.y - wy, pt.z - wz])
                row.append(label)
                writer.writerow(row)
                count += 1

            total += count
            print(f'  [{label}]  {count}/{len(images)} extracted')

print(f'\n✅ Done — {total} rows saved')
print(f'   Skipped: {skipped}')

  [A]  200/200 extracted
  [B]  200/200 extracted
  [C]  200/200 extracted
  [D]  200/200 extracted
  [DEL]  200/200 extracted
  [E]  200/200 extracted
  [F]  200/200 extracted
  [G]  200/200 extracted
  [H]  200/200 extracted
  [I]  200/200 extracted
  [J]  200/200 extracted
  [K]  200/200 extracted
  [L]  200/200 extracted
  [M]  198/200 extracted
  [N]  197/200 extracted
  [NOTHING]  200/200 extracted
  [O]  200/200 extracted
  [P]  200/200 extracted
  [Q]  200/200 extracted
  [R]  200/200 extracted
  [S]  200/200 extracted
  [SPACE]  193/200 extracted
  [T]  200/200 extracted
  [U]  200/200 extracted
  [V]  199/200 extracted
  [W]  199/200 extracted
  [X]  200/200 extracted
  [Y]  200/200 extracted
  [Z]  200/200 extracted

✅ Done — 5786 rows saved
   Skipped: 14


In [ ]:
# Cell 4 — Verify class distribution
import pandas as pd

df = pd.read_csv('/content/fingerspell_landmarks.csv')
print(f'Total samples: {len(df)}')
print(f'Classes ({df["label"].nunique()}): {sorted(df["label"].unique())}')
print('\nSamples per class:')
print(df['label'].value_counts().sort_index().to_string())

low = df['label'].value_counts()[df['label'].value_counts() < 50]
if len(low) > 0:
    print(f'\n⚠️  Low sample classes: {list(low.index)}')
else:
    print('\n✅ All classes look good')

Total samples: 5786
Classes (29): ['A', 'B', 'C', 'D', 'DEL', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'NOTHING', 'O', 'P', 'Q', 'R', 'S', 'SPACE', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']

Samples per class:
label
A          200
B          200
C          200
D          200
DEL        200
E          200
F          200
G          200
H          200
I          200
J          200
K          200
L          200
M          198
N          197
NOTHING    200
O          200
P          200
Q          200
R          200
S          200
SPACE      193
T          200
U          200
V          199
W          199
X          200
Y          200
Z          200

✅ All classes look good


In [ ]:
# Cell 5 — Train Random Forest
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

X = df.iloc[:, :-1].values
y = df.iloc[:,  -1].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Training...')
model = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f'\n✅ Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%')
print(classification_report(y_test, y_pred))

Training...

✅ Accuracy: 99.05%
              precision    recall  f1-score   support

           A       1.00      0.97      0.99        40
           B       0.95      1.00      0.98        40
           C       1.00      1.00      1.00        40
           D       1.00      1.00      1.00        40
         DEL       1.00      0.97      0.99        40
           E       1.00      0.95      0.97        40
           F       1.00      0.97      0.99        40
           G       0.97      0.97      0.97        40
           H       0.98      1.00      0.99        40
           I       1.00      1.00      1.00        40
           J       1.00      0.97      0.99        40
           K       0.98      1.00      0.99        40
           L       1.00      1.00      1.00        40
           M       1.00      1.00      1.00        40
           N       1.00      1.00      1.00        39
     NOTHING       1.00      0.97      0.99        40
           O       0.97      0.97      0.97      

In [ ]:
# Cell 6 — Save model files
import pickle

with open('/content/fingerspell_model.pkl', 'wb') as f:
    pickle.dump(model, f)

labels = list(model.classes_)
with open('/content/fingerspell_labels.pkl', 'wb') as f:
    pickle.dump(labels, f)

print('✅ Saved!')
print('Labels:', labels)

✅ Saved!
Labels: ['A', 'B', 'C', 'D', 'DEL', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'NOTHING', 'O', 'P', 'Q', 'R', 'S', 'SPACE', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']


In [ ]:
# Cell 7 — Download both files
from google.colab import files
files.download('/content/fingerspell_model.pkl')
files.download('/content/fingerspell_labels.pkl')
print('✅ Done! Replace the old .pkl files in your project folder')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Done! Replace the old .pkl files in your project folder
